## Preprocess Data the Male Training Data

In [ ]:
import pandas as pd

# File paths (updated for Extension_4.0)
base_path = '<LOCAL_PATH>'

# Define the datasets
datasets = {
    '(Train)': 'unaffected_combined.csv',
    '(Test)': 'affected_combined.csv'
}

# Define function to print unique IDs, status, and fix counts
def print_unique_ids_and_status(df, label):
    ids = sorted(df['id'].unique())
    print(f"\n📄 {label} — {len(ids)} unique IDs:")
    print(ids)

    # CWD Positive
    positive_ids = sorted(df.loc[df['CWD_status'] == 'positive', 'id'].unique())
    print(f"  ➡ CWD Positive IDs ({len(positive_ids)}): {positive_ids}")

    # CWD Negative
    negative_ids = sorted(df.loc[df['CWD_status'] == 'negative', 'id'].unique())
    print(f"  ➡ CWD Negative IDs ({len(negative_ids)}): {negative_ids}")

    # Fix counts per ID
    fix_counts = df['id'].value_counts().sort_index()
    print("\n📊 Number of GPS fixes per ID:")
    print(fix_counts.to_string())

    # Optional: summary stats
    print(f"\n   ➤ Fix count summary: min={fix_counts.min()}, max={fix_counts.max()}, median={fix_counts.median()}, mean={fix_counts.mean():.1f}")

def run_datetime_eda(df, label):
    print(f"\n🧪 EDA on `datetime` for {label}")

    # Check for missing
    null_count = df['datetime'].isnull().sum()
    print(f"  ➤ Null values in `datetime`: {null_count}")

    # Convert to datetime
    df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
    coercion_count = df['datetime'].isnull().sum() - null_count
    if coercion_count > 0:
        print(f"  ⚠️ {coercion_count} additional nulls after conversion (possibly bad formats)")

    # Check range
    min_time = df['datetime'].min()
    max_time = df['datetime'].max()
    print(f"  ➤ Overall date range: {min_time} to {max_time}")

    # Duration per ID
    id_group = df.groupby('id')['datetime']
    durations = (id_group.max() - id_group.min()).dt.total_seconds() / (60*60*24)
    print(f"  ➤ Duration (days) across IDs: min={durations.min():.2f}, max={durations.max():.2f}, median={durations.median():.2f}")

    # Continuity: time between rows per ID
    discontinuous_ids = []
    for _id, group in df.sort_values('datetime').groupby('id'):
        times = group['datetime'].dropna()
        deltas = times.diff().dt.total_seconds().dropna()
        if (deltas == 0).all():
            discontinuous_ids.append(_id)

    if discontinuous_ids:
        print(f"  ⚠️ {len(discontinuous_ids)} IDs have no time variation (i.e., all timestamps are identical):")
        print(f"     {discontinuous_ids[:10]}{' ...' if len(discontinuous_ids) > 10 else ''}")
    else:
        print(f"  ✅ All IDs have time variation.")

    return df

# Load and process each dataset
for label, file_name in datasets.items():
    path = f"{base_path}/{file_name}"
    df = pd.read_csv(path)
    print_unique_ids_and_status(df, label)
    df = run_datetime_eda(df, label)


## Winter Dataset First - Remove 'area_square' from traininga and testing

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 100% experiment: leak-free preprocessing with ID-disjoint splits + train-only fit
# ──────────────────────────────────────────────────────────────────────────────
import os, random
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler

# ── REPRODUCIBILITY ───────────────────────────────────────────────────────────
os.environ["PYTHONHASHSEED"] = "0"
random.seed(13)
np.random.seed(13)

# ── CONFIG ─────────────────────────────────────────────────────────────────────
base       = Path("<LOCAL_PATH>")

# inputs: training pool is CWD-free; test is CWD-positive
train_raw  = base / "unaffected_combined.csv"   # negatives only (still guard below)
test_raw   = base / "affected_combined.csv"     # positives (mixed OK; we only transform)

# outputs
out_train  = base / "train_processed_100.csv"
out_val    = base / "val_processed_100.csv"
out_test   = base / "test_processed_100.csv"

enc_path   = base / "encoder_100.pkl"
sca_path   = base / "scaler_100.pkl"
feat_path  = base / "feature_columns_100.pkl"
split_path = base / "split_ids_100.pkl"

to_drop = [
    'x_2','x_1','x_2_2','transition_to_migratory','id_1','phases',
    'migration_class_grouped','hour','movement_group','r2n_smoothed',
    'suspect_jump','extreme_jump','SATS','rel.angle',
    'utm_easting','utm_northing','dx','dy','dt','cum_dist','dist_per_hr'
]

num_feats = [
    'velo','abs_rel_angle','sinuosity_28pt',
    'rolling_abs_rel_angle_mean','rolling_abs_rel_angle_sd',
    'rolling_abs_rel_angle_skew','rolling_abs_rel_angle_kurt',
    'rolling_velo_mean','rolling_velo_sd','rolling_velo_skew','rolling_velo_kurt',
    'low','est','high'
]

cat_feats  = ['season']
meta_feats = ['pkey','id','datetime']   # ← added pkey here

# ── UTILITIES ──────────────────────────────────────────────────────────────────
def basic_cleanup(df: pd.DataFrame) -> pd.DataFrame:
    """Drop unused cols, enforce numeric types, keep pkey intact, preserve order."""
    df = df.drop(columns=to_drop, errors='ignore').copy()
    
    # Preserve original datetime string AND a parsed version
    df['datetime_raw'] = df['datetime']   # keep raw untouched
    df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
    
    for c in num_feats:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    
    df = df.dropna(subset=num_feats + cat_feats).copy()
    
    # Sort by id + parsed datetime (if valid), fallback to pkey for stability
    df = df.sort_values(['id', 'datetime', 'pkey']).reset_index(drop=True)
    return df

def transform_with_artifacts(df, encoder, scaler, feature_columns):
    """Apply fitted encoder/scaler; return DataFrame with pkey + meta + features."""
    cat_mat = encoder.transform(df[cat_feats])
    num_mat = scaler.transform(df[num_feats])
    cat_names = encoder.get_feature_names_out(cat_feats).tolist()
    X = np.hstack([num_mat, cat_mat])
    feat_df = pd.DataFrame(X, columns=num_feats + cat_names)

    feat_df = feat_df.reindex(columns=feature_columns, fill_value=0.0)
    out = pd.concat([df[meta_feats].reset_index(drop=True),
                     feat_df.reset_index(drop=True)], axis=1)
    return out

# ── 1) LOAD TRAIN POOL ────────────────────────────────────────────────────────
df_train_pool = pd.read_csv(train_raw)
df_train_pool = df_train_pool[df_train_pool['CWD_status'] == "negative"].copy()
df_train_pool = basic_cleanup(df_train_pool)

# ── 2) ID-DISJOINT SPLIT ──────────────────────────────────────────────────────
ids = df_train_pool['id'].unique()
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=13)
train_idx, val_idx = next(gss.split(df_train_pool, groups=df_train_pool['id']))

train_ids = df_train_pool.iloc[train_idx]['id'].unique()
val_ids   = df_train_pool.iloc[val_idx]['id'].unique()

df_train = df_train_pool[df_train_pool['id'].isin(train_ids)].copy()
df_val   = df_train_pool[df_train_pool['id'].isin(val_ids)].copy()

# ── 3) FIT ENCODER & SCALER ON TRAIN ONLY ─────────────────────────────────────
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
scaler  = MinMaxScaler()

encoder.fit(df_train[cat_feats])
scaler.fit(df_train[num_feats])

cat_names = encoder.get_feature_names_out(cat_feats).tolist()
feature_columns = num_feats + cat_names

# ── 4) TRANSFORM SPLITS ──────────────────────────────────────────────────────
df_train_out = transform_with_artifacts(df_train, encoder, scaler, feature_columns)
df_val_out   = transform_with_artifacts(df_val,   encoder, scaler, feature_columns)

# ── 5) TEST ───────────────────────────────────────────────────────────────────
df_test_raw = pd.read_csv(test_raw)
df_test_raw = basic_cleanup(df_test_raw)
df_test_out = transform_with_artifacts(df_test_raw, encoder, scaler, feature_columns)

# ── 6) SAVE ARTIFACTS ─────────────────────────────────────────────────────────
joblib.dump(encoder, enc_path)
joblib.dump(scaler,  sca_path)
joblib.dump(feature_columns, feat_path)
joblib.dump(
    {"train_ids": sorted(map(int, set(train_ids))),
     "val_ids":   sorted(map(int, set(val_ids))),
     "test_ids":  sorted(map(int, set(df_test_out['id'].unique())))},
    split_path
)

df_train_out.to_csv(out_train, index=False)
df_val_out.to_csv(out_val, index=False)
df_test_out.to_csv(out_test, index=False)

print("✅ Preprocessing complete with pkey preserved.")
print("Train IDs:", sorted(set(train_ids)))
print("Val IDs:",   sorted(set(val_ids)))
print("Test IDs:",  sorted(df_test_out['id'].unique().tolist()))



In [ ]:
# ── Quick EDA on processed splits ─────────────────────────────────────────────
import pandas as pd

def quick_eda(path, name):
    df = pd.read_csv(path, parse_dates=["datetime"], infer_datetime_format=True)
    
    print(f"\n📊 {name} Split")
    print(f"  → Rows: {len(df)}")
    print(f"  → Unique IDs: {df['id'].nunique()}")
    
    # temporal range
    min_dt, max_dt = df['datetime'].min(), df['datetime'].max()
    print(f"  → Temporal range: {min_dt} → {max_dt}")
    
    # per-ID summary
    id_summary = df.groupby("id")["datetime"].agg(["min", "max", "count"])
    print("  → Per-ID counts (first 5 rows):")
    print(id_summary.head())
    return id_summary

train_summary = quick_eda(out_train, "Train")
val_summary   = quick_eda(out_val, "Validation")
test_summary  = quick_eda(out_test, "Test")

# Optional: Save summaries for manual inspection
train_summary.to_csv(base/"train_id_summary.csv")
val_summary.to_csv(base/"val_id_summary.csv")
test_summary.to_csv(base/"test_id_summary.csv")


In [ ]:
import tensorflow as tf
print(tf.__version__)


In [ ]:
import pandas as pd
print(pd.__version__)


In [ ]:
import numpy as np
print(np.__version__)

In [ ]:
import sklearn
print(sklearn.__version__)


## Train the Autoencoder

In [ ]:
import tensorflow as tf

# Check GPU availability
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print(tf.config.list_physical_devices('GPU'))

# Set memory growth to avoid memory allocation issues
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.experimental.list_logical_devices('GPU')
        print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
    except RuntimeError as e:
        print(e)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Autoencoder training USING CUSTOM LOSS + flexible thresholding
# ──────────────────────────────────────────────────────────────────────────────
import os, random
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras import layers, Model, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# ===== Reproducibility =====
os.environ["PYTHONHASHSEED"] = "0"
random.seed(13); np.random.seed(13)
tf.keras.utils.set_random_seed(13)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

# ===== Config / Paths =====
base = "<LOCAL_PATH>"
train_path = f"{base}/train_processed_100.csv"
val_path   = f"{base}/val_processed_100.csv"   # negatives-only val (from preprocessing)
test_path  = f"{base}/test_processed_100.csv"

feat_cols_path = f"{base}/feature_columns_100.pkl"
model_path     = f"{base}/autoencoder_model.h5"
threshold_path = f"{base}/reconstruction_threshold.pkl"
history_path   = f"{base}/training_history.npy"
ckpt_path      = f"{base}/ae_ckpt.weights.h5"

# ===== Threshold policy (choose one) =====
THRESHOLD_POLICY = "train_lowest_p95"   # "train_lowest_p95" | "train_quantile" | "val_row_fpr" | "val_id_flag_rate" | "mad"

# Params for each policy
LOWEST_FRAC   = 0.75      # for "train_lowest_p95" 
LOWEST_Q      = 90        # percentile on the lowest fraction
TRAIN_Q       = 0.90      # for "train_quantile": e.g., 90th percentile of all train errors
VAL_ROW_FPR   = 0.20      # for "val_row_fpr": allow 20% of NEG val rows to flag → lower θ, more flags
VAL_ID_RATE   = 0.20      # for "val_id_flag_rate": allow 20% of NEG val IDs to flag at least 1 row
MIN_ROWS_PER_ID = 1       # ignore super-short IDs when computing ID-level rate
MAD_K         = 3.0       # for "mad"


# ===== Load features with frozen column order =====
feature_columns = joblib.load(feat_cols_path)

def load_X(path):
    df = pd.read_csv(path)
    if "datetime" in df.columns:
        df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
    X = df[feature_columns].values.astype(np.float32)
    meta = df[["id"]].copy()
    if "datetime" in df.columns:
        meta["datetime"] = df["datetime"].values
    return X, meta, df

X_train, meta_train, df_train_raw = load_X(train_path)
X_val,   meta_val,   df_val_raw   = load_X(val_path)
X_test,  meta_test,  df_test_raw  = load_X(test_path)

# ===== tf.data pipelines =====
batch_size = 128
AUTOTUNE = tf.data.AUTOTUNE

def make_ds(X, training=False):
    ds = tf.data.Dataset.from_tensor_slices((X, X))
    if training:
        ds = ds.shuffle(min(10000, len(X)), seed=13, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size, drop_remainder=False).cache().prefetch(AUTOTUNE)
    return ds

train_ds = make_ds(X_train, training=True)
val_ds   = make_ds(X_val,   training=False)

# ===== Model =====
input_dim = X_train.shape[1]
reg = regularizers.l2(1e-4)

inp = layers.Input(shape=(input_dim,))
x = layers.Dense(32, activation='relu', kernel_regularizer=reg)(inp)
x = layers.Dense(16, activation='relu', kernel_regularizer=reg)(x)
z = layers.Dense(4,  activation='relu', kernel_regularizer=reg, name="bottleneck")(x)
y = layers.Dense(16, activation='relu', kernel_regularizer=reg)(z)
y = layers.Dense(32, activation='relu', kernel_regularizer=reg)(y)
out = layers.Dense(input_dim, activation='linear')(y)
autoencoder = Model(inp, out, name="autoencoder")

# === Custom loss ===
LAMBDA_DEV = 0.5                      # ← your choice
LOSS_SCALE = 1.0 + LAMBDA_DEV*input_dim  # α = 1 + λ d

def get_custom_loss(loss_scale=LOSS_SCALE):
    def custom_loss(y_true, y_pred):
        mse = tf.reduce_mean(tf.square(y_true - y_pred), axis=-1)  # mean over features
        return loss_scale * mse
    return custom_loss

loss_fn = get_custom_loss(LOSS_SCALE)
autoencoder.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=loss_fn)

# ===== Train =====
early_stop = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
ckpt = ModelCheckpoint(filepath=ckpt_path, save_weights_only=True, monitor="val_loss", save_best_only=True)

history = autoencoder.fit(
    train_ds, epochs=100, validation_data=val_ds, callbacks=[early_stop, ckpt], verbose=1
)

autoencoder.save(model_path)
np.save(history_path, history.history, allow_pickle=True)
print(f"✅ Saved model → {model_path}")

# ===== Errors on train/val (row-level, using SAME loss) =====
recon_train = autoencoder.predict(X_train, batch_size=batch_size, verbose=0)
train_errors = loss_fn(X_train, recon_train).numpy().astype(np.float64)

recon_val = autoencoder.predict(X_val, batch_size=batch_size, verbose=0)
val_errors = loss_fn(X_val, recon_val).numpy().astype(np.float64)

# ===== Threshold policies =====
def choose_threshold(policy: str):
    info = {"policy": policy}   # <— standard key

    if policy == "train_lowest_p95":
        n_low = int(len(train_errors) * LOWEST_FRAC)
        base = np.sort(train_errors)[:n_low]
        theta = float(np.percentile(base, LOWEST_Q))
        info.update({"lowest_frac": LOWEST_FRAC, "percentile": LOWEST_Q, "n_low": n_low})

    elif policy == "train_quantile":
        theta = float(np.quantile(train_errors, TRAIN_Q))
        info.update({"train_quantile": TRAIN_Q})

    elif policy == "val_row_fpr":
        q = 1.0 - float(VAL_ROW_FPR)
        theta = float(np.quantile(val_errors, q))
        info.update({"val_row_fpr": VAL_ROW_FPR, "val_quantile": q})

    elif policy == "val_id_flag_rate":
        q_grid = np.linspace(0.50, 0.999, 200)
        ids = df_val_raw["id"].values
        best_theta, best_rate, best_q = None, None, None
        for q in q_grid:
            th = np.quantile(val_errors, q)
            row_flag = val_errors > th
            id_any = pd.Series(row_flag).groupby(ids, sort=False).any()
            n_rows = pd.Series(1, index=ids).groupby(ids).sum()
            eligible = n_rows[n_rows >= MIN_ROWS_PER_ID].index
            rate = float(id_any.reindex(eligible).fillna(False).mean())
            if rate <= VAL_ID_RATE:
                best_theta, best_rate, best_q = float(th), rate, float(q)
                break
        if best_theta is None:
            best_q = float(q_grid[-1])
            best_theta = float(np.quantile(val_errors, best_q))
            row_flag = val_errors > best_theta
            id_any = pd.Series(row_flag).groupby(ids, sort=False).any()
            n_rows = pd.Series(1, index=ids).groupby(ids).sum()
            eligible = n_rows[n_rows >= MIN_ROWS_PER_ID].index
            best_rate = float(id_any.reindex(eligible).fillna(False).mean())
        theta = best_theta
        info.update({"val_id_rate_target": VAL_ID_RATE, "achieved_rate": best_rate, "val_quantile": best_q})

    elif policy == "mad":
        med = float(np.median(train_errors))
        mad = float(np.median(np.abs(train_errors - med)) + 1e-12)
        theta = med + MAD_K * 1.4826 * mad
        info.update({"median": med, "mad": mad, "k": MAD_K})

    else:
        raise ValueError(f"Unknown THRESHOLD_POLICY: {policy}")

    return theta, info

theta, th_info = choose_threshold(THRESHOLD_POLICY)

# Persist policy hyperparameters for reproducibility
if THRESHOLD_POLICY == "train_lowest_p95":
    th_info.update({"LOWEST_FRAC": float(LOWEST_FRAC), "LOWEST_Q": float(LOWEST_Q)})
elif THRESHOLD_POLICY == "train_quantile":
    th_info.update({"TRAIN_Q": float(TRAIN_Q)})
elif THRESHOLD_POLICY == "val_row_fpr":
    th_info.update({"VAL_ROW_FPR": float(VAL_ROW_FPR)})
elif THRESHOLD_POLICY == "val_id_flag_rate":
    th_info.update({"VAL_ID_RATE": float(VAL_ID_RATE), "MIN_ROWS_PER_ID": int(MIN_ROWS_PER_ID)})
elif THRESHOLD_POLICY == "mad":
    th_info.update({"MAD_K": float(MAD_K)})

th_info["effective_multiplier"] = float(LOSS_SCALE)  # alias for reviewers
print(f"[loss] λ={LAMBDA_DEV} | d={input_dim} | α=1+λ·d={LOSS_SCALE:.3f} (scaled-MSE)")


# ===== Diagnostics: how aggressive is θ? =====
def percent_rows_flagged(errors, theta):
    return float(np.mean(errors > theta))

train_row_rate = percent_rows_flagged(train_errors, theta)
val_row_rate   = percent_rows_flagged(val_errors, theta)

# ID-level on val (negatives)
val_ids = df_val_raw["id"].values
val_row_flag = val_errors > theta
val_id_any = pd.Series(val_row_flag).groupby(val_ids, sort=False).any()
val_n_rows = pd.Series(1, index=val_ids).groupby(val_ids).sum()
val_eligible = val_n_rows[val_n_rows >= MIN_ROWS_PER_ID].index
val_id_rate = float(val_id_any.reindex(val_eligible).fillna(False).mean())

print(f"📏 θ = {theta:.6g} | policy = {THRESHOLD_POLICY}")
print(f"   Train rows flagged: {100*train_row_rate:.1f}%")
print(f"   Val   rows flagged: {100*val_row_rate:.1f}%")
print(f"   Val     IDs flagged (≥1 row): {100*val_id_rate:.1f}% (min_rows={MIN_ROWS_PER_ID})")

# ===== Persist threshold + bookkeeping =====
# Diagnostics
# (unchanged, but print 'policy')
print(f"📏 θ = {theta:.6g} | policy = {THRESHOLD_POLICY}")

# Persist threshold + bookkeeping (use 'policy')
th_info.update({
    "policy": THRESHOLD_POLICY,
    "theta": float(theta),
    "lambda_dev": float(LAMBDA_DEV),
    "input_dim": int(input_dim),
    "loss_scale": float(LOSS_SCALE),  # α
    "loss_name": "scaled_mse_equivalent_to_mse_plus_lambda_sse",
    "train_rows_flagged": train_row_rate,
    "val_rows_flagged": val_row_rate,
    "val_ids_flagged": val_id_rate,
    "min_rows_per_id": int(MIN_ROWS_PER_ID)
})
joblib.dump(th_info, threshold_path)
print(f"✅ Threshold saved → {threshold_path}")


## Custom Loss Function Explanation

The custom loss function can be described as:

\begin{equation}
\text{Custom Loss}(y_{\text{true}}, y_{\text{reconstructed}}) = \text{MSE}(y_{\text{true}}, y_{\text{reconstructed}}) + \lambda_{\text{dev}} \times \text{Dev}(y_{\text{true}}, y_{\text{reconstructed}})
\end{equation}

Where:
- $\text{MSE}$ is the Mean Squared Error, representing the average squared difference between the actual values ($y_{\text{true}}$) and the reconstructed values ($y_{\text{reconstructed}}$) from the autoencoder.
- $\lambda_{\text{dev}}$ is a weight that adjusts the relative importance of the deviation penalty, allowing the model to focus more on larger reconstruction errors which could indicate anomalies.
- $\text{Dev}$ represents a penalty term for deviations in the reconstruction of the data, acting as a proxy for detecting anomalies. It is calculated as an additional term based on the magnitude of the reconstruction error.





## Preprocessing Testing Data and Making Predictions on Anomalous Mule Deer

In [ ]:
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras.models import load_model
from pathlib import Path

# ── CONFIG ─────────────────────────────────────────────────────────────────────
base = Path("<LOCAL_PATH>")
val_csv         = base / "val_processed_100.csv"
model_path      = base / "autoencoder_model.h5"
threshold_path  = base / "reconstruction_threshold.pkl"   # contains θ + loss params from training
feat_cols_path  = base / "feature_columns_100.pkl"      # enforce column order
param_out_path  = base / "id_rule_params.pkl"

USE_MIXED_VAL = False  # your val is negatives-only per preprocessing

# preregistered operating point (choose & freeze before test)
FROZEN_COUNT     = 75
FROZEN_FRAC      = 0.75
MIN_ROWS_PER_ID  = 1          # consider 100 if short IDs are noisy
AGGREGATION_RULE = "OR"       # freeze: "OR", "AND", "FRAC_ONLY", or "COUNT_ONLY"

# if you ever use mixed val with labels, these grids are considered a priori
COUNTS = [50, 100, 150, 200]
FRACS  = [0.80, 0.90, 0.925]
POS_IDS_VAL = []   # only used when USE_MIXED_VAL=True

# ── LOAD MODEL / FEATURES / TRAIN ARTIFACT ────────────────────────────────────
feature_columns = joblib.load(feat_cols_path)
autoencoder = load_model(model_path, compile=False)

# load training artifact with θ + loss params
th_info   = joblib.load(threshold_path)
theta     = float(th_info["theta"])
policy    = th_info.get("policy", "unknown")

# reconstruct scaled-MSE loss consistently
lambda_dev = float(th_info.get("lambda_dev", 0.5))
input_dim  = int(th_info.get("input_dim", len(feature_columns)))
loss_scale = float(th_info.get("loss_scale", 1.0 + lambda_dev * input_dim))

def get_custom_loss(loss_scale=loss_scale):
    def custom_loss(y_true, y_pred):
        mse = tf.reduce_mean(tf.square(y_true - y_pred), axis=-1)
        return loss_scale * mse
    return custom_loss

loss_fn = get_custom_loss(loss_scale)
print(f"Using TRAIN-derived θ = {theta:.6f} (policy={policy}) | λ={lambda_dev} | d={input_dim} | α={loss_scale:.6g}")

# ── LOAD VAL AND COMPUTE ROW ERRORS (FROZEN FEATURE ORDER) ────────────────────
df_val = pd.read_csv(val_csv)
if "datetime" in df_val.columns:
    df_val["datetime"] = pd.to_datetime(df_val["datetime"], errors="coerce")

X_val     = df_val[feature_columns].values.astype(np.float32)
recon_val = autoencoder.predict(X_val, batch_size=128, verbose=0)
val_err   = loss_fn(X_val, recon_val).numpy().astype(np.float64)

rows_val = pd.DataFrame({
    "id": df_val["id"].values,
    "datetime": df_val["datetime"] if "datetime" in df_val.columns else pd.NaT,
    "reconstruction_error": val_err
})
rows_val["row_flag"] = rows_val["reconstruction_error"] > theta

# ── AGGREGATE BY ID ───────────────────────────────────────────────────────────
g         = rows_val.groupby("id", sort=False)
n_rows    = g.size()
n_flags   = g["row_flag"].sum()
flag_frac = n_flags / n_rows

# consistent eligible set (evaluation/selection must use the same gate)
eligible_idx = n_rows[n_rows >= MIN_ROWS_PER_ID].index
n_rows    = n_rows.reindex(eligible_idx).astype(int)
n_flags   = n_flags.reindex(eligible_idx).astype(int)
flag_frac = flag_frac.reindex(eligible_idx).astype(float)

# ── SELECT (COUNT, FRAC) IF MIXED VAL; OTHERWISE FREEZE A PRIORI ─────────────
if USE_MIXED_VAL:
    pos_val = set(POS_IDS_VAL) & set(eligible_idx)
    if len(pos_val) == 0:
        raise ValueError("No positive IDs present in eligible validation set.")

    best = None; best_key = None
    for c in COUNTS:
        for f in FRACS:
            idx_count = set(n_flags[n_flags >= c].index)
            idx_frac  = set(flag_frac[flag_frac >= f].index)

            if   AGGREGATION_RULE == "AND":       pred = idx_count & idx_frac
            elif AGGREGATION_RULE == "FRAC_ONLY": pred = idx_frac
            elif AGGREGATION_RULE == "COUNT_ONLY":pred = idx_count
            else:                                 pred = idx_count | idx_frac  # OR

            recall = len(pred & pos_val) / len(pos_val)
            key = (recall, c, f)  # maximize recall; tie-break to stricter (higher c,f)
            if (best is None) or (key > best_key):
                best = {"count": int(c), "frac": float(f), "val_recall": float(recall)}
                best_key = key

    COUNT, FRAC = best["count"], best["frac"]
    print(f"Frozen ID rule from validation (positives-only, {AGGREGATION_RULE}): "
          f"COUNT={COUNT} | FRAC={FRAC} | recall={best['val_recall']:.3f}")
else:
    COUNT, FRAC = int(FROZEN_COUNT), float(FROZEN_FRAC)
    print(f"Using pre-registered ID rule ({AGGREGATION_RULE}; negatives-only val): "
          f"COUNT={COUNT} | FRAC={FRAC} | min_rows={MIN_ROWS_PER_ID}")

# ── SAVE FROZEN PARAMS FOR TEST ───────────────────────────────────────────────
frozen = {
    "theta": float(theta),
    "count": int(COUNT),
    "frac": float(FRAC),
    "lambda_dev": float(lambda_dev),
    "input_dim": int(input_dim),
    "loss_scale": float(loss_scale),
    "policy": policy,
    "aggregation_rule": AGGREGATION_RULE,  # ← frozen & explicit
    "val_meta": {
        "min_rows_per_id": int(MIN_ROWS_PER_ID),
        "use_mixed_val": bool(USE_MIXED_VAL)
    }
}
joblib.dump(frozen, param_out_path)
print(f"✅ Saved frozen params to: {param_out_path}")


In [ ]:

import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras.models import load_model
from pathlib import Path

# ── PATHS ──────────────────────────────────────────────────────────────────────
base = Path("<LOCAL_PATH>")
test_feat_path  = base / "test_processed_100.csv"
model_path      = base / "autoencoder_model.h5"
frozen_param_p  = base / "id_rule_params.pkl"           # produced in validation (frozen params)
feat_cols_path  = base / "feature_columns_100.pkl"      # frozen feature order

rows_out_csv    = str(test_feat_path).replace(".csv", "_rows_with_flags_FROZEN.csv")
ids_out_csv     = str(test_feat_path).replace(".csv", "_id_summary_FROZEN.csv")
residuals_out   = str(test_feat_path).replace(".csv", "_rows_with_recon_and_residuals.csv")

# (Optional) test labels for evaluation
actual_cwd_instances_test     = [1, 2, 6, 8, 11, 12, 15, 16, 17, 18, 19, 20, 21, 23, 24, 25,
                                 30, 31, 32, 35, 36, 37, 40, 47, 48, 53, 57, 58, 61, 62, 64,
                                 65, 67, 68, 69, 70, 71, 72, 73, 74, 79, 80, 83, 90, 91, 92,
                                 93, 94, 96, 102, 105, 107, 113, 114, 116, 121, 122, 125, 130, 141]
unaffected_cwd_instances_test = []

# ── LOAD ARTIFACTS ────────────────────────────────────────────────────────────
feature_columns = joblib.load(feat_cols_path)           # list[str]
frozen = joblib.load(frozen_param_p)                    # dict from validation

theta  = float(frozen["theta"])
COUNT  = int(frozen["count"])
FRAC   = float(frozen["frac"])
policy = frozen.get("policy", "unknown")

# loss consistency (scaled-MSE, α = 1 + λ·d)
lambda_dev = float(frozen.get("lambda_dev", 0.5))
input_dim  = int(frozen.get("input_dim", len(feature_columns)))
loss_scale = float(frozen.get("loss_scale", 1.0 + lambda_dev * input_dim))

AGGREGATION_RULE = str(frozen.get("aggregation_rule", "OR")).upper()
MIN_ROWS_PER_ID  = int(frozen.get("val_meta", {}).get("min_rows_per_id", 1))

print(f"Using frozen params → θ={theta:.6f} | COUNT={COUNT} | FRAC={FRAC} | "
      f"λ={lambda_dev} | d={input_dim} | α={loss_scale:.3f} | MIN_ROWS={MIN_ROWS_PER_ID} | "
      f"policy={policy} | agg_rule={AGGREGATION_RULE}")

def get_custom_loss(loss_scale=loss_scale):
    def custom_loss(y_true, y_pred):
        mse = tf.reduce_mean(tf.square(y_true - y_pred), axis=-1)
        return loss_scale * mse
    return custom_loss

loss_fn = get_custom_loss(loss_scale)
autoencoder = load_model(model_path, compile=False)

# ── LOAD TEST + COMPUTE RECONS + RESIDUALS ────────────────────────────────────
df = pd.read_csv(test_feat_path)
if "datetime" in df.columns:
    df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")

X_test = df[feature_columns].values.astype(np.float32)
recon  = autoencoder.predict(X_test, batch_size=128, verbose=0)

row_err = loss_fn(X_test, recon).numpy().astype(np.float64)
row_flag = row_err > theta

# absolute residuals per feature
residuals = np.abs(X_test - recon)
df_resid = pd.DataFrame(residuals, columns=[f"{c}_resid" for c in feature_columns])
df_recon = pd.DataFrame(recon, columns=[f"{c}_recon" for c in feature_columns])

rows_full = pd.concat([
    df.reset_index(drop=True)[["id", "datetime"]],
    pd.DataFrame({"reconstruction_error": row_err, "row_flag": row_flag}),
    df_recon, df_resid
], axis=1)

rows_full.to_csv(residuals_out, index=False)
print(f"✅ Saved per-row reconstructions + residuals → {residuals_out}")

# ── CONTINUE WITH AGGREGATION / IDS SUMMARY (unchanged) ───────────────────────
g        = pd.DataFrame({"row_flag": row_flag, "err": row_err, "id": df["id"]}).groupby("id", sort=False)
n_rows   = g.size()
n_flags  = g["row_flag"].sum()
flag_fr  = n_flags / n_rows
id_max   = g["err"].max()
id_mean  = g["err"].mean()

eligible_idx = n_rows[n_rows >= MIN_ROWS_PER_ID].index
n_flags_elig = n_flags.reindex(eligible_idx)
flag_fr_elig = flag_fr.reindex(eligible_idx)

if AGGREGATION_RULE == "AND":
    pred_ids = set(n_flags_elig[n_flags_elig >= COUNT].index) & set(flag_fr_elig[flag_fr_elig >= FRAC].index)
elif AGGREGATION_RULE == "FRAC_ONLY":
    pred_ids = set(flag_fr_elig[flag_fr_elig >= FRAC].index)
elif AGGREGATION_RULE == "COUNT_ONLY":
    pred_ids = set(n_flags_elig[n_flags_elig >= COUNT].index)
else:  # OR
    pred_ids = set(n_flags_elig[n_flags_elig >= COUNT].index) | set(flag_fr_elig[flag_fr_elig >= FRAC].index)

pred_ids = sorted(pred_ids)
print("Predicted positive IDs:", pred_ids)

rows = pd.DataFrame({
    "id": df["id"].values,
    "datetime": df["datetime"] if "datetime" in df.columns else pd.NaT,
    "reconstruction_error": row_err,
    "row_flag": row_flag
})
rows.to_csv(rows_out_csv, index=False)

# … the rest of your ID-level summary code here …


# Rule-aligned and component scores (for plots & AUC)
eps = 1e-12
count_norm_all = n_flags / max(COUNT, eps)
frac_norm_all  = flag_fr / max(FRAC, eps)

if AGGREGATION_RULE == "AND":
    score_all = np.minimum(count_norm_all, frac_norm_all)
elif AGGREGATION_RULE == "FRAC_ONLY":
    score_all = frac_norm_all
elif AGGREGATION_RULE == "COUNT_ONLY":
    score_all = count_norm_all
else:  # OR
    score_all = np.maximum(count_norm_all, frac_norm_all)

ids_df = pd.DataFrame({
    "id": n_rows.index,
    "n_rows": n_rows.values,
    "n_flags": n_flags.reindex(n_rows.index).values,
    "flag_frac": flag_fr.reindex(n_rows.index).values,
    "rule_aligned_score": score_all.reindex(n_rows.index).values,
    "frac_norm_score": frac_norm_all.reindex(n_rows.index).values,
    "count_norm_score": count_norm_all.reindex(n_rows.index).values,
    "id_max_error": id_max.reindex(n_rows.index).values,
    "id_mean_error": id_mean.reindex(n_rows.index).values,
    "eligible": [1 if i in set(eligible_idx) else 0 for i in n_rows.index],
    "is_positive_id": [1 if i in pred_ids else 0 for i in n_rows.index]
})
ids_df.to_csv(ids_out_csv, index=False)
print(f"✅ Wrote: {rows_out_csv}\n✅ Wrote: {ids_out_csv}")

# ── (OPTIONAL) TEST EVALUATION — eligible set only, rule-aligned score ────────
pos_set = set(actual_cwd_instances_test)
neg_set = set(unaffected_cwd_instances_test)

eval_ids   = sorted((pos_set | neg_set) & set(eligible_idx))
n_pos_eval = sum(i in pos_set for i in eval_ids)
n_neg_eval = sum(i in neg_set for i in eval_ids)
print(f"Eval on eligible IDs only → n_eval={len(eval_ids)} (pos={n_pos_eval}, neg={n_neg_eval}; min_rows={MIN_ROWS_PER_ID})")

if eval_ids and (n_pos_eval > 0 or n_neg_eval > 0):
    y_true = np.array([1 if i in pos_set else 0 for i in eval_ids])
    y_pred = np.array([1 if i in pred_ids else 0 for i in eval_ids])

    if (y_true == 1).all():
        TP = np.sum(y_pred == 1)
        FN = np.sum(y_pred == 0)
        FP = 0
        TN = 0

        sensitivity = TP / (TP + FN) if (TP + FN) else 0.0
        precision   = TP / (TP + FP) if (TP + FP) else 0.0
        accuracy    = (TP + TN) / (TP + FN + TN + FP) if (TP + FN + TN + FP) else 0.0
        f1_score    = 2 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) else 0.0
        prevalence  = 1.0
        tss         = np.nan

        print(f"TP = {TP}, FN = {FN}")
        print(f"Sensitivity     (TPR) = {sensitivity:.3f}")
        print(f"Precision              = {precision:.3f}")
        print(f"Accuracy               = {accuracy:.3f}")
        print(f"F1 Score               = {f1_score:.3f}")
        print(f"Prevalence Identified  = {prevalence:.3f}")
        print(f"TSS                   = N/A (no negatives)")


In [ ]:
# Define the actual CWD instances and unaffected CWD instances
actual_cwd_instances = [1, 2, 6, 8, 11, 12, 15, 16, 17, 18, 19, 20, 21, 23, 24, 25, 30, 31, 32, 35, 36, 37, 40, 47, 48, 53, 57, 58, 61, 62, 64, 65, 67, 68, 69, 70, 71, 72, 73, 74, 79, 80, 83, 90, 91, 92, 93, 94, 96, 102, 105, 107, 113, 114, 116, 121, 122, 125, 130, 141]
unaffected_cwd_instances = []

# You already have high_anomaly_ids from Step 1
print("High anomaly IDs:", pred_ids)

# Convert high_anomaly_ids to a set for easier comparison
high_deviation_ids = set(pred_ids)

# Calculate True Positives (TP), False Negatives (FN), False Positives (FP), and True Negatives (TN)
TP = len([ID for ID in actual_cwd_instances if ID in pred_ids])
FN = len([ID for ID in actual_cwd_instances if ID not in pred_ids])
FP = len([ID for ID in unaffected_cwd_instances if ID in pred_ids])
TN = len([ID for ID in unaffected_cwd_instances if ID not in pred_ids])

# Calculate False Positive Rate (FPR), False Negative Rate (FNR), and Accuracy
FPR = FP / (FP + TN) if (FP + TN) > 0 else 0
FNR = FN / (FN + TP) if (FN + TP) > 0 else 0
accuracy = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0

# Print the results
print(f"True Positives (TP): {TP}")
print(f"False Negatives (FN): {FN}")
print(f"False Positives (FP): {FP}")
print(f"True Negatives (TN): {TN}")
print(f"False Positive Rate (FPR): {FPR:.2f}")
print(f"False Negative Rate (FNR): {FNR:.2f}")
print(f"Accuracy: {accuracy:.2f}")





In [ ]:
id_summary = df.groupby("id").agg({
    "datetime": ["min", "max", "count"],
    "id": "count"
}).droplevel(0, axis=1).rename(columns={"min": "start", "max": "end", "count": "n_rows"})
id_summary["duration_hours"] = (id_summary["end"] - id_summary["start"]).dt.total_seconds() / 3600
id_summary.sort_values("duration_hours", ascending=False).head(10)

id_summary.head()



In [ ]:
# ===== Binned (by ID duration in days) metrics & counts — SAME 2×3 COMPOSITION =====
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# 0) Load full test and get per-ID duration (days)
df_full = pd.read_csv(test_feat_path)
time_col = "date_time" if "date_time" in df_full.columns else ("datetime" if "datetime" in df_full.columns else None)
if time_col is None:
    raise KeyError("No 'date_time' or 'datetime' column in test CSV.")
df_full[time_col] = pd.to_datetime(df_full[time_col], errors="coerce")

id_start = df_full.groupby("id")[time_col].min()
id_end   = df_full.groupby("id")[time_col].max()
id_duration_days = (id_end - id_start).dt.total_seconds() / 86400.0

# 1) Quantile bins on DAYS (equal-count), robust to ties/small N
N_BINS = 7
q = min(N_BINS, max(1, id_duration_days.dropna().nunique()))
cats, edges = pd.qcut(id_duration_days, q=q, retbins=True, duplicates='drop')

def _label(left, right):
    L = int(np.floor(left)); R = int(np.ceil(right))
    return f"{L}–{R} days" if R > L else f"{L} days"

intervals  = cats.cat.categories
bin_labels = [_label(iv.left, iv.right) for iv in intervals]
cats = cats.cat.rename_categories(bin_labels)

# Map bin label to each ID and attach to ids_df
id_to_bin = pd.Series(cats.values, index=id_duration_days.index).to_dict()
ids_df = ids_df.copy()
ids_df["bin"] = ids_df["id"].map(id_to_bin)

# 2) Helpers
def wilson_ci(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n
    denom = 1 + z*z/n
    center = (p + z*z/(2*n)) / denom
    half = z * np.sqrt((p*(1-p)/n) + (z*z/(4*n*n))) / denom
    return (center - half, center + half)

def counts_to_metrics(TN, FP, FN, TP):
    N = TP + TN + FP + FN
    acc  = (TP + TN) / N if N else np.nan
    prec = TP / (TP + FP) if (TP + FP) else np.nan
    sens = TP / (TP + FN) if (TP + FN) else np.nan
    spec = TN / (TN + FP) if (TN + FP) else np.nan
    tss  = sens + spec - 1 if np.isfinite(sens) and np.isfinite(spec) else np.nan
    acc_lo, acc_hi   = wilson_ci(TP + TN, N) if N else (np.nan, np.nan)
    prec_lo, prec_hi = wilson_ci(TP, TP + FP) if (TP + FP) else (np.nan, np.nan)
    sens_lo, sens_hi = wilson_ci(TP, TP + FN) if (TP + FN) else (np.nan, np.nan)
    spec_lo, spec_hi = wilson_ci(TN, TN + FP) if (TN + FP) else (np.nan, np.nan)
    tss_lo, tss_hi   = (sens_lo + spec_lo - 1, sens_hi + spec_hi - 1) if np.isfinite(sens_lo) and np.isfinite(spec_lo) else (np.nan, np.nan)
    return dict(accuracy=acc, accuracy_lo=acc_lo, accuracy_hi=acc_hi,
                precision=prec, precision_lo=prec_lo, precision_hi=prec_hi,
                tss=tss, tss_lo=tss_lo, tss_hi=tss_hi)

# 3) Build per-bin summary using your *non-CV* predictions (pred_ids) + label lists
#    (If you’re in CV/ensemble mode, swap to ids_df['true_label_id'] and ids_df['id_pred'].)
summary = []
bin_order = bin_labels  # preserve quantile order

for b in bin_order:
    sub = ids_df[(ids_df["bin"] == b) & (ids_df["eligible"] == 1)]
    eval_ids = sub["id"].tolist()
    if len(eval_ids) == 0:
        summary.append({"bin": b, "n_eval": 0, "TP":0, "FP":0, "FN":0, "TN":0,
                        "accuracy": np.nan, "accuracy_lo": np.nan, "accuracy_hi": np.nan,
                        "precision": np.nan, "precision_lo": np.nan, "precision_hi": np.nan,
                        "tss": np.nan, "tss_lo": np.nan, "tss_hi": np.nan})
        continue

    # Ground truth & predictions from your lists
    pos_set = set(actual_cwd_instances_test)
    y_true = np.array([1 if i in pos_set else 0 for i in eval_ids])
    y_pred = np.array([1 if i in pred_ids else 0 for i in eval_ids])

    TN, FP, FN, TP = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    met = counts_to_metrics(TN, FP, FN, TP)

    summary.append({"bin": b, "n_eval": len(eval_ids), "TP":TP, "FP":FP, "FN":FN, "TN":TN, **met})

bin_df = pd.DataFrame(summary)
bin_df["bin"] = pd.Categorical(bin_df["bin"], categories=bin_order, ordered=True)
bin_df = bin_df.sort_values("bin")

print("\nPer-bin counts (eligible IDs):")
print(bin_df[["bin","n_eval","TP","FP","FN","TN"]].to_string(index=False))

# 4) Plot — SAME 2×3 COMPOSITION
def plot_bars_three(ddd: pd.DataFrame, title: str, fname: str):
    if ddd.empty:
        print("⚠️ Nothing to plot.")
        return
    ddd = ddd.copy()
    labels = ddd["bin"].astype(str).tolist()
    x = np.arange(len(labels), dtype=float)

    # Ensure columns exist
    for col in ["accuracy","accuracy_lo","accuracy_hi",
                "precision","precision_lo","precision_hi",
                "tss","tss_lo","tss_hi","TP","FP","FN","TN"]:
        if col not in ddd.columns: ddd[col] = np.nan

    fig, ax = plt.subplots(2, 3, figsize=(18, 10), sharex=False)

    panels = [
        ("accuracy","accuracy_lo","accuracy_hi","Accuracy", 0.0, 1.0, "#cfe1f2"),
        ("precision","precision_lo","precision_hi","Precision", 0.0, 1.0, "#cfe1f2"),
        ("tss","tss_lo","tss_hi","TSS", -1.0, 1.0, "#dcd3f2"),
    ]
    for i, (m, lo, hi, ttl, ymin, ymax, color) in enumerate(panels):
        center = ddd[m].to_numpy(float)
        lo_arr = center - ddd[lo].to_numpy(float)
        hi_arr = ddd[hi].to_numpy(float) - center
        msk = np.isfinite(center) & np.isfinite(lo_arr) & np.isfinite(hi_arr)

        # placeholders so bins always show
        ax[0,i].bar(x, np.nan_to_num(center, nan=0.0), color="#eeeeee", edgecolor="black",
                    linewidth=0.6, width=0.8, zorder=1)
        # real bars + error
        ax[0,i].bar(x[msk], center[msk], color=color, edgecolor="black",
                    linewidth=0.6, width=0.8, zorder=2)
        if np.any(msk):
            ax[0,i].errorbar(x[msk], center[msk], yerr=np.vstack([lo_arr[msk], hi_arr[msk]]),
                             fmt="none", ecolor="black", capsize=5, elinewidth=1.2, zorder=3)

        ax[0,i].set_title(ttl); ax[0,i].set_ylim(ymin, ymax)
        ax[0,i].set_xlim(-0.5, len(x)-0.5)
        ax[0,i].grid(axis="y", linestyle="--", alpha=0.35)
        ax[0,i].set_ylabel("Score")

    # Bottom row: TP/FP/FN; TN on separate axis
    for j, cm_m, col in [(0,"TP","#f6c0c0"), (1,"FP","#f6c0c0"), (2,"FN","#f6c0c0")]:
        vals = ddd[cm_m].fillna(0).to_numpy(int)
        ax[1,j].bar(x, vals, color=col, edgecolor="black", linewidth=0.6)
        ax[1,j].set_title(cm_m); ax[1,j].set_ylabel("Count")
        ax[1,j].set_xlim(-0.5, len(x)-0.5)
        ax[1,j].grid(axis="y", linestyle="--", alpha=0.35)

    fig.delaxes(ax[1,2])
    tn_ax = fig.add_subplot(2, 3, 6)
    tn_ax.bar(x, ddd["TN"].fillna(0).to_numpy(int), color="#cfeccf", edgecolor="black", linewidth=0.6)
    tn_ax.set_title("TN"); tn_ax.set_ylabel("Count")
    tn_ax.set_xlim(-0.5, len(x)-0.5); tn_ax.grid(axis="y", linestyle="--", alpha=0.35)

    for a in [ax[1,0], ax[1,1], tn_ax]:
        a.set_xticks(x); a.set_xticklabels(labels, rotation=30, ha="right")
    for a in ax[0,:]:
        a.set_xticks(x); a.set_xticklabels([])

    plt.suptitle("Full Holdout Experiment: Metrics & Confusion Counts by Duration Bin", fontsize=16, y=0.98)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    out_png = base / "ae_metrics_bins_by_duration_same_composition.png"
    out_pdf = base / "ae_metrics_bins_by_duration_same_composition.pdf"
    plt.savefig(out_png, bbox_inches="tight"); plt.savefig(out_pdf, bbox_inches="tight")
    print(f"📊 Saved: {out_png}\n📊 Saved: {out_pdf}")
    plt.show(); plt.close(fig)

plot_bars_three(bin_df, "Full Holdout Experiment: Metrics & Counts by Duration Bin", "bins_by_duration_same_composition")

In [ ]:
id_summary = df.groupby("id").agg({
    "datetime": ["min", "max", "count"],
    "id": "count"
}).droplevel(0, axis=1).rename(columns={"min": "start", "max": "end", "count": "n_rows"})
id_summary["duration_hours"] = (id_summary["end"] - id_summary["start"]).dt.total_seconds() / 3600
id_summary.sort_values("duration_hours", ascending=False).head(10)

id_summary.head()


In [ ]:
# ===== AE Feature Contributions — holdout-friendly (manuscript-ready, with TP/FN) =====
import numpy as np
import pandas as pd
from collections import defaultdict
from pathlib import Path
import joblib

# ----------------------------
# Config (point to your 100% holdout experiment)
# ----------------------------
base   = Path("<LOCAL_PATH>")
outdir = base / "ae_holdout_results"
outdir.mkdir(parents=True, exist_ok=True)

# Processed files & artifacts produced by your preprocessing pipeline
PROC_TEST = base / "test_processed_100.csv"
SCA_PATH  = base / "scaler_100.pkl"
FEAT_PATH = base / "feature_columns_100.pkl"

# If you want to predict reconstruction on-the-fly:
TRY_PREDICT_RECON = True   # set False if you define 'recon' upstream
BATCH = 256                # batch size for predict()

# Row-level flagging (if no 'row_flag' column exists in test)
ROW_FLAG_TAU  = None       # absolute threshold on total residual (sum of sq err) e.g., 3.5
ROW_FLAG_PCTL = 95.0       # percentile threshold fallback (top X% rows flagged)

# ID aggregation → predicted positives (used to form TP/FN etc.)
MIN_ROWS_PER_ID = 10
AGGREGATION_RULE = "OR"    # {"OR","AND","COUNT_ONLY","FRAC_ONLY"}
COUNT = 5
FRAC  = 0.05

# Ground-truth on holdout:
#   - If your test is all-positive and lacks labels ⇒ set ASSUME_TEST_ALL_POS=True
#   - If test has per-row or per-id label you trust (e.g., 'CWD_label') ⇒ set to False
ASSUME_TEST_ALL_POS = True

# ----------------------------
# Load processed test & feature list
# ----------------------------
df_test    = pd.read_csv(PROC_TEST)
feat_names = joblib.load(FEAT_PATH)  # exact order used for training
if not all(c in df_test.columns for c in feat_names):
    missing = [c for c in feat_names if c not in df_test.columns]
    raise KeyError(f"Processed test missing features: {missing[:10]}{' ...' if len(missing)>10 else ''}")

X_test   = df_test[feat_names].to_numpy(dtype=np.float32)
ids_rows = df_test["id"].to_numpy()
has_row_flag = "row_flag" in df_test.columns  # optional, from upstream pipeline

# ----------------------------
# Reconstructions & residuals
# ----------------------------
try:
    _ = recon  # use precomputed recon if provided
except NameError:
    if not TRY_PREDICT_RECON:
        raise ValueError("No 'recon' provided and TRY_PREDICT_RECON is False.")
    try:
        _ = autoencoder
    except NameError:
        raise ValueError("To predict 'recon' on the fly, define 'autoencoder' in the environment.")
    recon = autoencoder.predict(X_test, batch_size=BATCH, verbose=0)

recon = np.asarray(recon, float)
if X_test.shape != recon.shape:
    raise ValueError(f"Shape mismatch: X_test {X_test.shape} vs recon {recon.shape}")

residual_sq = (X_test - recon) ** 2     # (N_rows, n_features)
resid_total = residual_sq.sum(axis=1)   # per-row total residual (τ)

# ----------------------------
# Build the 'rows' frame for masks
# ----------------------------
rows = pd.DataFrame({"id": ids_rows})
if has_row_flag:
    rows["row_flag"] = df_test["row_flag"].astype(bool).to_numpy()
else:
    if ROW_FLAG_TAU is not None:
        rows["row_flag"] = resid_total > ROW_FLAG_TAU
    else:
        thr = np.nanpercentile(resid_total, ROW_FLAG_PCTL)
        rows["row_flag"] = resid_total >= thr

mask_all  = np.ones(len(rows), dtype=bool)
mask_flag = rows["row_flag"].to_numpy(bool)
mask_eval = mask_all.copy()  # whole holdout

# ----------------------------
# Per-feature contribution helper (→ %)
# ----------------------------
def contrib_pct(rsq, mask):
    if rsq.size == 0 or not np.any(mask):
        return np.full(rsq.shape[1], np.nan, dtype=float)
    m = np.nanmean(rsq[mask], axis=0)
    s = np.nansum(m)
    return (m / s) * 100.0 if s > 0 else np.full(rsq.shape[1], np.nan, dtype=float)

contrib_all   = contrib_pct(residual_sq, mask_all)
contrib_flag  = contrib_pct(residual_sq, mask_flag)
contrib_eval  = contrib_pct(residual_sq, mask_eval)

# ----------------------------
# Erraticness across IDs (median per-feature SD in %-points)
# ----------------------------
id_to_rows = defaultdict(list)
for r, i in enumerate(rows["id"].to_numpy()):
    id_to_rows[i].append(r)

def per_id_contrib_stats(mask):
    per_id = []
    for i, idxs in id_to_rows.items():
        idxs = np.asarray(idxs, dtype=int)
        idxs = idxs[mask[idxs]]
        if idxs.size == 0:
            continue
        m = np.nanmean(residual_sq[idxs], axis=0)
        s = np.nansum(m)
        per_id.append((i, (m / s * 100.0) if s > 0 else np.full(residual_sq.shape[1], np.nan)))
    if not per_id:
        return dict(median_feature_sd_pp=np.nan, n_ids=0)
    M = np.vstack([v for _, v in per_id])  # (n_ids, n_features)
    std_per_feat_pp = np.nanstd(M, axis=0)
    return dict(median_feature_sd_pp=float(np.nanmedian(std_per_feat_pp)), n_ids=len(per_id))

err_all  = per_id_contrib_stats(mask_all)
err_flag = per_id_contrib_stats(mask_flag)
err_eval = per_id_contrib_stats(mask_eval)

# ----------------------------
# Simple concentration proxies
# ----------------------------
def topk_share_pct(arr, k=3):
    a = np.asarray(arr, dtype=float)
    a = a[np.isfinite(a)]
    if a.size == 0: return np.nan
    a_sorted = np.sort(a)[::-1]
    return float(np.sum(a_sorted[:min(k, a_sorted.size)]))

top3_all,  top5_all  = topk_share_pct(contrib_all, 3), topk_share_pct(contrib_all, 5)
top3_flag, top5_flag = topk_share_pct(contrib_flag, 3), topk_share_pct(contrib_flag, 5)
top3_eval, top5_eval = topk_share_pct(contrib_eval, 3), topk_share_pct(contrib_eval, 5)

# ----------------------------
# Per-feature table
# ----------------------------
df_contrib = pd.DataFrame({
    "feature":    feat_names,
    "All_%":      contrib_all,
    "Flagged_%":  contrib_flag,
    "Eval_%":     contrib_eval,
})
df_contrib["Delta_Flagged_minus_All_pp"] = df_contrib["Flagged_%"] - df_contrib["All_%"]

print("\n=== AE Feature Contributions (% of reconstruction error) — Holdout ===")
print(df_contrib.sort_values("Flagged_%", ascending=False).head(15).to_string(index=False,
      float_format=lambda x: f"{x:.2f}"))

summary_tbl = pd.DataFrame([
    {"Group":"All rows",    "Top3_share_%": top3_all,  "Top5_share_%": top5_all,  "Median_feat_SD_pp": err_all["median_feature_sd_pp"],  "n_ids": err_all["n_ids"]},
    {"Group":"Flagged rows","Top3_share_%": top3_flag, "Top5_share_%": top5_flag, "Median_feat_SD_pp": err_flag["median_feature_sd_pp"], "n_ids": err_flag["n_ids"]},
    {"Group":"Eval IDs",    "Top3_share_%": top3_eval, "Top5_share_%": top5_eval, "Median_feat_SD_pp": err_eval["median_feature_sd_pp"], "n_ids": err_eval["n_ids"]},
])
summary_tbl["ΔFlagged−All_Top3_pp"] = summary_tbl.loc[1,"Top3_share_%"] - summary_tbl.loc[0,"Top3_share_%"]
summary_tbl["ΔFlagged−All_Top5_pp"] = summary_tbl.loc[1,"Top5_share_%"] - summary_tbl.loc[0,"Top5_share_%"]
summary_tbl["ΔFlagged−All_SD_pp"]   = summary_tbl.loc[1,"Median_feat_SD_pp"] - summary_tbl.loc[0,"Median_feat_SD_pp"]

print("\n=== Simple concentration & variability summary (holdout) ===")
print(summary_tbl.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

# ----------------------------
# Category roll-up (movement story)
# ----------------------------
def categorize(fname: str) -> str:
    f = fname.lower()
    if "velo" in f: return "velocity"
    if "angle" in f: return "turning-angle"
    if "sinu" in f:  return "sinuosity"
    if "skew" in f or "kurt" in f: return "distributional-moment"
    if "rolling" in f or "mean" in f or "sd" in f: return "rolling-statistic"
    if f in {"low","est","high"} or "model" in f: return "model-derived"
    if "season" in f: return "season"
    return "other"

cat = pd.DataFrame({
    "feature":  feat_names,
    "category": [categorize(f) for f in feat_names],
    "All_%":    contrib_all,
    "Flagged_%":contrib_flag,
    "Eval_%":   contrib_eval
})
cat_roll = (cat.groupby("category")[["All_%","Flagged_%","Eval_%"]]
              .sum().sort_values("Flagged_%", ascending=False))
cat_roll["Delta_Flagged_minus_All_pp"] = cat_roll["Flagged_%"] - cat_roll["All_%"]

print("\n=== Category roll-up (% of recon error) — Holdout ===")
print(cat_roll.to_string(float_format=lambda x: f"{x:.1f}"))

# ----------------------------
# Build TP / FP / FN / TN partitions (ID-level)
# ----------------------------
# 1) Ground-truth ID sets
if ("pos_set" in globals()) and ("neg_set" in globals()):
    # Use externally provided sets if you already defined them
    pos_set, neg_set = set(pos_set), set(neg_set)
else:
    if (not ASSUME_TEST_ALL_POS) and ("CWD_label" in df_test.columns):
        # Use ID-level labels from df_test if present
        id_lab = df_test.groupby("id")["CWD_label"].max()
        pos_set = set(id_lab.index[id_lab == 1])
        neg_set = set(id_lab.index[id_lab == 0])
    else:
        # Fallback: holdout contains only positives
        pos_set = set(df_test["id"].unique())
        neg_set = set()

# 2) Aggregate row_flags → predicted positive IDs
def aggregate_ids_from_rowflags(row_flags_bool, ids_series,
                                MIN_ROWS_PER_ID=MIN_ROWS_PER_ID,
                                AGGREGATION_RULE=AGGREGATION_RULE,
                                COUNT=COUNT, FRAC=FRAC):
    tmp = pd.DataFrame({"id": ids_series.values, "row_flag": row_flags_bool.astype(int)})
    g = tmp.groupby("id", sort=False)
    n_rows = g.size()
    n_flags = g["row_flag"].sum()
    flag_fr = n_flags / n_rows

    eligible_idx = n_rows[n_rows >= MIN_ROWS_PER_ID].index
    n_flags_elig = n_flags.reindex(eligible_idx)
    flag_fr_elig = flag_fr.reindex(eligible_idx)

    if AGGREGATION_RULE == "AND":
        pred = set(n_flags_elig[n_flags_elig >= COUNT].index) & set(flag_fr_elig[flag_fr_elig >= FRAC].index)
    elif AGGREGATION_RULE == "FRAC_ONLY":
        pred = set(flag_fr_elig[flag_fr_elig >= FRAC].index)
    elif AGGREGATION_RULE == "COUNT_ONLY":
        pred = set(n_flags_elig[n_flags_elig >= COUNT].index)
    else:  # "OR"
        pred = set(n_flags_elig[n_flags_elig >= COUNT].index) | set(flag_fr_elig[flag_fr_elig >= FRAC].index)
    return sorted(pred), eligible_idx

pred_ids = globals().get("pred_ids", None)
if pred_ids is None:
    pred_ids, eligible_idx = aggregate_ids_from_rowflags(rows["row_flag"].to_numpy(bool), rows["id"])
else:
    pred_ids = sorted(set(pred_ids))
    eligible_idx = rows["id"].unique()

# 3) Confusion partitions on eligible IDs
eval_ids = sorted((pos_set | neg_set) & set(eligible_idx))
TP_ids = [i for i in eval_ids if (i in pos_set) and (i in pred_ids)]
FP_ids = [i for i in eval_ids if (i in neg_set) and (i in pred_ids)]
FN_ids = [i for i in eval_ids if (i in pos_set) and (i not in pred_ids)]
TN_ids = [i for i in eval_ids if (i in neg_set) and (i not in pred_ids)]

rid = rows["id"]
m_tp = rid.isin(TP_ids).to_numpy()
m_fp = rid.isin(FP_ids).to_numpy()
m_fn = rid.isin(FN_ids).to_numpy()
m_tn = rid.isin(TN_ids).to_numpy()

# ----------------------------
# Per-class contribution tables
# ----------------------------
def table_from_mask(mask, label):
    pct = contrib_pct(residual_sq, mask)
    return (pd.DataFrame({"feature": feat_names, f"{label}_%": pct})
              .sort_values(f"{label}_%", ascending=False).reset_index(drop=True))

tp_tbl = table_from_mask(m_tp, "TP")
fp_tbl = table_from_mask(m_fp, "FP")
fn_tbl = table_from_mask(m_fn, "FN")
tn_tbl = table_from_mask(m_tn, "TN")

# Print top-10s with Δ vs All (guard empty classes)
all_series = pd.Series(contrib_all, index=feat_names, name="All_%")
def top_with_delta(df, label, k=10):
    if df[f"{label}_%"].notna().any():
        out = (df.head(k)
                 .merge(all_series.reset_index().rename(columns={"index":"feature"}),
                        on="feature", how="left"))
        out[f"Delta_{label}_minus_All_pp"] = out[f"{label}_%"] - out["All_%"]
        print(f"\n=== Top {k} features — {label} ===")
        print(out.to_string(index=False, float_format=lambda x: f"{x:.2f}"))
    else:
        print(f"\n=== Top {k} features — {label} ===\n(no rows)")

top_with_delta(tp_tbl,"TP"); top_with_delta(fp_tbl,"FP")
top_with_delta(fn_tbl,"FN"); top_with_delta(tn_tbl,"TN")

def table_from_mask(mask, label):
    pct = contrib_pct(residual_sq, mask)
    return (pd.DataFrame({"feature": feat_names, f"{label}_%": pct})
              .sort_values(f"{label}_%", ascending=False).reset_index(drop=True))

def top_with_delta(df, label, k=10):
    if df[f"{label}_%"].notna().any():
        out = (df.head(k)
                 .merge(all_series.reset_index().rename(columns={"index":"feature"}),
                        on="feature", how="left"))
        out[f"Delta_{label}_minus_All_pp"] = out[f"{label}_%"] - out["All_%"]
        print(f"\n=== Top {k} features — {label} ===")
        print(out.to_string(index=False, float_format=lambda x: f"{x:.2f}"))


# Save per-class CSVs
tp_tbl.to_csv(outdir / "ae_holdout_feat_contrib_tp.csv", index=False)
fp_tbl.to_csv(outdir / "ae_holdout_feat_contrib_fp.csv", index=False)
fn_tbl.to_csv(outdir / "ae_holdout_feat_contrib_fn.csv", index=False)
tn_tbl.to_csv(outdir / "ae_holdout_feat_contrib_tn.csv", index=False)

# 🔄 Save ALL per-class features (not top-10)
for label, df in {"TP": tp_tbl, "FP": fp_tbl, "FN": fn_tbl, "TN": tn_tbl}.items():
    if df is not None and not df.empty:
        # Merge with All_% and compute delta for every feature
        merged = df.merge(all_series.reset_index().rename(columns={"index": "feature"}),
                          on="feature", how="left")
        merged[f"Delta_{label}_minus_All_pp"] = merged[f"{label}_%"] - merged["All_%"]
        merged.to_csv(outdir / f"ae_all_features_{label}.csv", index=False)
        print(f"✅ Saved full feature list for {label}: {outdir}/ae_all_features_{label}.csv")


# ----------------------------
# Movement–residual linkage (inverse-scaling numeric features only)
# ----------------------------
try:
    scaler = joblib.load(SCA_PATH)
    if not hasattr(scaler, "inverse_transform"):
        raise TypeError("Loaded scaler has no inverse_transform")
    num_cols = [c for c in feat_names if c in getattr(scaler, "feature_names_in_", [])]
    cat_cols = [c for c in feat_names if c not in num_cols]

    X_num = df_test[num_cols].to_numpy()
    X_cat = df_test[cat_cols].to_numpy() if cat_cols else np.empty((len(df_test), 0))
    X_num_raw = scaler.inverse_transform(X_num)

    df_raw = pd.DataFrame(np.hstack([X_num_raw, X_cat]), columns=num_cols + cat_cols)
    units_label = "raw units"
except Exception as e:
    print(f"[warn] Using scaled units — {e}")
    df_raw = df_test[feat_names].copy()
    units_label = "scaled units"

# Add IDs + row flags
df_raw["id"] = rows["id"].values
df_raw["row_flag"] = rows["row_flag"].values

# Collapsed movement metrics
df_raw["speed"]   = df_raw.filter(like="velo").mean(axis=1)
df_raw["turning"] = df_raw.filter(like="angle").mean(axis=1)
df_raw["sinu"]    = df_raw.filter(like="sinu").mean(axis=1)

def group_summary(mask, label):
    tmp = df_raw.loc[mask]
    return pd.Series({
        "n_rows":       int(len(tmp)),
        "mean_speed":   float(tmp["speed"].mean())   if len(tmp) else np.nan,
        "mean_turning": float(tmp["turning"].mean()) if len(tmp) else np.nan,
        "mean_sinu":    float(tmp["sinu"].mean())    if len(tmp) else np.nan
    }, name=label)

# Always provide All/Flagged; add TP/FN/FP/TN if non-empty
summ_pieces = [
    group_summary(mask_all, "All"),
    group_summary(mask_flag, "Flagged"),
]
if m_tp.any(): summ_pieces.append(group_summary(m_tp, "TP"))
if m_fn.any(): summ_pieces.append(group_summary(m_fn, "FN"))
if m_fp.any(): summ_pieces.append(group_summary(m_fp, "FP"))
if m_tn.any(): summ_pieces.append(group_summary(m_tn, "TN"))

summaries = pd.concat(summ_pieces, axis=1).T

# %diff vs TP for the groups that exist and if TP has rows
if ("TP" in summaries.index) and (summaries.loc["TP", "n_rows"] > 0):
    for col in ["mean_speed","mean_turning","mean_sinu"]:
        base_val = summaries.loc["TP", col]
        for grp in summaries.index:
            if grp == "TP": continue
            summaries.loc[grp, f"{col}_vsTP_%diff"] = (
                (summaries.loc[grp, col] - base_val) / base_val * 100
                if (pd.notna(base_val) and base_val != 0) else np.nan
            )

print(f"\n=== Movement summaries by class ({units_label}) — Holdout ===")
print(summaries.round(3))
summaries.to_csv(outdir / f"ae_holdout_movement_summaries_by_class_{'raw' if units_label=='raw units' else 'scaled'}.csv")

# Direct per-variable summaries
vars_of_interest = [
    "abs_rel_angle", "est", "low", "high",
    "rolling_abs_rel_angle_kurt", "rolling_abs_rel_angle_mean",
    "rolling_abs_rel_angle_sd", "rolling_abs_rel_angle_skew",
    "rolling_velo_kurt", "rolling_velo_mean", "rolling_velo_sd",
    "rolling_velo_skew", "sinuosity_28pt", "velo"
]
present = [v for v in vars_of_interest if v in df_raw.columns]

def group_summary_vars(mask, label):
    tmp = df_raw.loc[mask, present]
    stats = {f"{v}_mean": tmp[v].mean() for v in present}
    stats.update({f"{v}_sd": tmp[v].std() for v in present})
    stats["n_rows"] = len(tmp)
    return pd.Series(stats, name=label)

summ_vars_pieces = [
    group_summary_vars(mask_all,  "All"),
    group_summary_vars(mask_flag, "Flagged"),
]
if m_tp.any(): summ_vars_pieces.append(group_summary_vars(m_tp, "TP"))
if m_fn.any(): summ_vars_pieces.append(group_summary_vars(m_fn, "FN"))
if m_fp.any(): summ_vars_pieces.append(group_summary_vars(m_fp, "FP"))
if m_tn.any(): summ_vars_pieces.append(group_summary_vars(m_tn, "TN"))

summaries_vars = pd.concat(summ_vars_pieces, axis=1).T

# diffs vs TP (only if TP exists)
if "TP" in summaries_vars.index and summaries_vars.loc["TP","n_rows"] > 0:
    for v in present:
        base = summaries_vars.loc["TP", f"{v}_mean"]
        for grp in summaries_vars.index:
            if grp == "TP": continue
            summaries_vars.loc[grp, f"{v}_diff_vsTP"] = summaries_vars.loc[grp, f"{v}_mean"] - base
            summaries_vars.loc[grp, f"{v}_pctdiff_vsTP"] = (
                (summaries_vars.loc[grp, f"{v}_mean"] - base) / base * 100
                if (pd.notna(base) and base != 0) else np.nan
            )

print("\n=== Per-variable movement summaries — Holdout ===")
print(summaries_vars.round(3))
summaries_vars.to_csv(outdir / f"ae_holdout_movement_summaries_by_variable_{'raw' if units_label=='raw units' else 'scaled'}.csv")

# Residual correlation with movement metrics
movement_links = pd.DataFrame({
    "resid_total": resid_total,
    "speed":       df_raw["speed"],
    "turning":     df_raw["turning"],
    "sinu":        df_raw["sinu"],
}).corr()

print("\n=== Correlation of movement variables with total reconstruction error ===")
print(movement_links.round(3))
movement_links.to_csv(outdir / f"ae_holdout_movement_residual_correlations_{'raw' if units_label=='raw units' else 'scaled'}.csv")

# ----------------------------
# Save the main contribution tables
# ----------------------------
(df_contrib.sort_values("Flagged_%", ascending=False)
          .to_csv(outdir / "ae_holdout_feature_contributions_by_feature.csv", index=False))
cat_roll.to_csv(outdir / "ae_holdout_feature_contributions_by_category.csv")
summary_tbl.to_csv(outdir / "ae_holdout_feature_contrib_summary_simple.csv", index=False)

print(f"\n💾 Saved tables to: {outdir}")
print(f"IDs — TP:{len(TP_ids)}  FN:{len(FN_ids)}  FP:{len(FP_ids)}  TN:{len(TN_ids)}  | Eligible:{len(set(eval_ids))}")


In [ ]:
# make_early_late_vs_neg_pct_heatmap_RAW.py
import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib as mpl, seaborn as sns
from pathlib import Path
import joblib

# === Paths ===
BASE        = Path("<LOCAL_PATH>")
TRAIN_PATH  = BASE / "train_processed_100.csv"
VAL_PATH    = BASE / "val_processed_100.csv"
TEST_PATH   = BASE / "test_processed_100.csv"
SCALER_PATH = BASE / "scaler_100.pkl"     # <- fit on training (scaled -> raw)
OUT_DIR     = BASE / "figures_early_late"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SAVE_BASE   = OUT_DIR / "early_late_vs_neg_pct_heatmap_RAW_New"

# === IDs & features ===
ACTUAL_POS_IDS = [1,2,6,8,11,12,15,16,17,18,19,20,21,23,24,25,30,31,32,35,36,37,40,47,48,53,57,58,
                  61,62,64,65,67,68,69,70,71,72,73,74,79,80,83,90,91,92,93,94,96,102,105,107,113,
                  114,116,121,122,125,130,141]
BECAME_POS_IDS = [6,11,19,20,25,32,40,48,62,74,105,122,125,47,53,73,80,92,107,116,135]  
# Desired column order: early−neg, late−neg, early−late
comp_order = ["Early − Neg", "Late − Neg", "Early − Late"]

# Load data (scaled)
train, val, test = (pd.read_csv(p) for p in [TRAIN_PATH, VAL_PATH, TEST_PATH])

# Load scaler and define features from scaler (ensures exact order & transform)
scaler   = joblib.load(SCALER_PATH)
FEATURES = list(scaler.feature_names_in_)  # guarantees alignment to the fitted scaler

# Keep only required columns (id + FEATURES), same order as scaler
def keep_cols(df): return df[["id"] + FEATURES].copy()
train, val, test = map(keep_cols, (train, val, test))

# Inverse-transform to RAW units
def inv(df):
    X_raw = scaler.inverse_transform(df[FEATURES].to_numpy())
    out = df[["id"]].copy()
    out[FEATURES] = X_raw
    return out
train_raw, val_raw, test_raw = map(inv, (train, val, test))

# Negatives = train+val in RAW units
neg_raw = pd.concat([train_raw, val_raw], ignore_index=True)

# Early/Late present in TEST
test_ids  = set(test_raw["id"].unique())

# Early = converters (subset of ACTUAL_POS_IDS)
early_ids = sorted(set(BECAME_POS_IDS) & test_ids)

# Late = already positive at first capture (remaining positives)
late_ids  = sorted(set(ACTUAL_POS_IDS) & test_ids - set(early_ids))

# Group means (RAW units)
def group_mean(df, ids): 
    return df[df["id"].isin(ids)][FEATURES].mean(numeric_only=True)

mu_neg   = group_mean(neg_raw,  neg_raw["id"].unique().tolist())
mu_early = group_mean(test_raw, early_ids)   # converters
mu_late  = group_mean(test_raw, late_ids)   # already-positive

# % differences in RAW units: 100*(A−B)/max(|B|, floor)
BASE_FLOOR = 1e-8  # tiny floor to avoid div-by-zero in raw space
def pct_diff(A, B, floor=BASE_FLOOR):
    A, B = np.asarray(A, float), np.asarray(B, float)
    den  = np.maximum(np.abs(B), floor)
    return 100.0*(A - B)/den

all_df = pd.DataFrame({
    "feature": FEATURES,
    "Late − Neg":   pct_diff(mu_late.values,  mu_neg.values),
    "Early − Neg":  pct_diff(mu_early.values, mu_neg.values),
    "Early − Late": pct_diff(mu_early.values, mu_late.values),
})

# Pivot to feature x comparison
heatmap_df = all_df.set_index("feature")[comp_order]

# Pretty ordering & labels
pretty_order = [
    # Velocity group
    "velo", "rolling_velo_mean", "rolling_velo_sd",
    "rolling_velo_skew", "rolling_velo_kurt",
    # Direction/turning group
    "abs_rel_angle", "rolling_abs_rel_angle_mean",
    "rolling_abs_rel_angle_sd", "rolling_abs_rel_angle_skew",
    "rolling_abs_rel_angle_kurt",
    # Other metrics
    "sinuosity_28pt",
    # Space-use group
    "est", "low", "high"
]
heatmap_df = heatmap_df.reindex([r for r in pretty_order if r in heatmap_df.index])
pretty_map = {
    "velo": "Velocity",
    "rolling_velo_mean": "Rolling Velocity Mean",
    "rolling_velo_sd": "Rolling Velocity SD",
    "rolling_velo_skew": "Rolling Velocity Skew",
    "rolling_velo_kurt": "Rolling Velocity Kurtosis",
    "abs_rel_angle": "Absolute Relative Angle",
    "rolling_abs_rel_angle_mean": "Rolling Abs Rel Angle Mean",
    "rolling_abs_rel_angle_sd": "Rolling Abs Rel Angle SD",
    "rolling_abs_rel_angle_skew": "Rolling Abs Rel Angle Skew",
    "rolling_abs_rel_angle_kurt": "Rolling Abs Rel Angle Kurtosis",
    "sinuosity_28pt": "Sinuosity",
    "est": "Median Space Usage",
    "low": "Low CI Space Usage",
    "high": "High CI Space Usage",
}
heatmap_df.index = [pretty_map.get(r, r) for r in heatmap_df.index]

# === Plot (narrower columns, consistent with other heatmaps) ===
save_base = str(BASE / "figures_early_late" / "early_late_vs_neg_pct_heatmap_scaled_allrows")
Path(save_base).parent.mkdir(parents=True, exist_ok=True)

# Get actual data bounds (for reporting only)
# --- Use true data range ---
data_min = float(np.nanmin(heatmap_df.to_numpy(dtype=float)))
data_max = float(np.nanmax(heatmap_df.to_numpy(dtype=float)))

print(f"Data range: {data_min:.1f} to {data_max:.1f}")

# --- Suggested rounded bounds (per reviewer) ---
# Example: data_min=-47.4 -> -48, data_max=22.3 -> 23
vmin = np.floor(data_min)   # -> -48.0
vmax = np.ceil(data_max)    # ->  23.0

# If you want to FORCE exactly -48 to 23 no matter what:
# vmin, vmax = -48, 23

print(f"Scale range (rounded): {vmin:.0f} to {vmax:.0f}")

# --- size from matrix shape: tighter columns ---
n_rows, n_cols = heatmap_df.shape
COL_W, ROW_H, CBAR_W, MARG_W = 3.05, 0.50, 0.55, 1.20

# let it get wider than 8.6
fig_w = np.clip(COL_W*n_cols + CBAR_W + MARG_W, 7.4, 11.5)
fig_h = np.clip(ROW_H*n_rows + 1.6,            7.6, 10.8)

# --- Font sizes ---
AX_LABEL_FS   = 18
TICK_FS       = 14
ANNOT_FS      = 13
CBAR_LABEL_FS = 16
CBAR_TICK_FS  = 14

fig = plt.figure(figsize=(fig_w, fig_h), constrained_layout=True)
gs  = fig.add_gridspec(nrows=1, ncols=2, width_ratios=[18, 1.2])  # slightly wider colorbar
ax  = fig.add_subplot(gs[0, 0])
cax = fig.add_subplot(gs[0, 1])

# IMPORTANT: lock the normalization so colors & colorbar are consistent
norm = mpl.colors.TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)

hm = sns.heatmap(
    heatmap_df,
    cmap="RdBu_r",
    norm=norm,                 # <-- use norm instead of vmin/vmax/center trio
    annot=True, fmt=".1f",
    annot_kws={"fontsize": ANNOT_FS, "fontweight": "medium"},
    linewidths=0.6, linecolor="white",
    ax=ax, cbar=True, cbar_ax=cax
)

# --- Axes labels + ticks ---
ax.set_xlabel("Comparison", fontsize=AX_LABEL_FS)
ax.set_ylabel("Feature", fontsize=AX_LABEL_FS)
ax.tick_params(axis="x", labelsize=TICK_FS)
ax.tick_params(axis="y", labelsize=TICK_FS)

# --- Colorbar formatting: bigger fonts + nice ticks ---
cbar = hm.collections[0].colorbar
cbar.set_label("% Difference (Raw Units)", rotation=90, labelpad=16, fontsize=CBAR_LABEL_FS)
cbar.ax.tick_params(labelsize=CBAR_TICK_FS)

# Use "nice" ticks (rounded), NOT the exact max
nice_ticks = np.array([vmin, -30, -15, 0, 10, vmax], dtype=float)
nice_ticks = nice_ticks[(nice_ticks >= vmin) & (nice_ticks <= vmax)]
cbar.set_ticks(nice_ticks)
cbar.set_ticklabels([f"{t:.0f}%" for t in nice_ticks])

# Lock limits (extra safety)
cbar.mappable.set_clim(vmin, vmax)

# ✅ Add "Max = <true max>%" above the colorbar
true_max = float(np.nanmax(heatmap_df.values))  # e.g., 22.3
cbar.ax.text(
    0.5, 1.04, f"Max = {true_max:.1f}",
    transform=cbar.ax.transAxes,
    ha="center", va="bottom",
    fontsize=CBAR_TICK_FS, fontweight="bold"
)

cax.yaxis.set_label_position("right")
cax.yaxis.set_ticks_position("right")

fig.savefig(f"{save_base}.png", dpi=300, bbox_inches="tight")
fig.savefig(f"{save_base}.pdf", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)


In [ ]:
# make_figure5_absolute_key_params.py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import joblib

# =========================
# Paths
# =========================
BASE = Path("<LOCAL_PATH>")
TRAIN_PATH = BASE / "train_processed_100.csv"
VAL_PATH   = BASE / "val_processed_100.csv"
TEST_PATH  = BASE / "test_processed_100.csv"
SCALER_PATH = BASE / "scaler_100.pkl"

OUT_DIR = BASE / "figures_early_late"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SAVE_BASE = OUT_DIR / "Figure5_absolute_key_params"

# =========================
# IDs
# =========================
ACTUAL_POS_IDS = [
    1,2,6,8,11,12,15,16,17,18,19,20,21,23,24,25,30,31,32,35,36,37,40,47,48,53,57,58,
    61,62,64,65,67,68,69,70,71,72,73,74,79,80,83,90,91,92,93,94,96,102,105,107,113,
    114,116,121,122,125,130,141
]
BECAME_POS_IDS = [6,11,19,20,25,32,40,48,62,74,105,122,125,47,53,73,80,92,107,116,135]

# =========================
# Load scaled data
# =========================
train = pd.read_csv(TRAIN_PATH)
val   = pd.read_csv(VAL_PATH)
test  = pd.read_csv(TEST_PATH)

scaler = joblib.load(SCALER_PATH)
FEATURES = list(scaler.feature_names_in_)

def keep_cols(df):
    return df[["id"] + FEATURES].copy()

train = keep_cols(train)
val   = keep_cols(val)
test  = keep_cols(test)

# =========================
# Inverse-transform to raw units
# =========================
def inverse_to_raw(df: pd.DataFrame) -> pd.DataFrame:
    x_raw = scaler.inverse_transform(df[FEATURES].to_numpy())
    out = df[["id"]].copy()
    out[FEATURES] = x_raw
    return out

train_raw = inverse_to_raw(train)
val_raw   = inverse_to_raw(val)
test_raw  = inverse_to_raw(test)

# =========================
# Define groups
# =========================
neg_raw = pd.concat([train_raw, val_raw], ignore_index=True)

test_ids = set(test_raw["id"].unique())
early_ids = sorted(set(BECAME_POS_IDS) & test_ids)
late_ids  = sorted((set(ACTUAL_POS_IDS) & test_ids) - set(early_ids))

# =========================
# Choose key parameters
# Reviewer asked for "key movement parameters" on absolute scale
# =========================
key_features = [
    "velo",
    "rolling_velo_mean",
    "rolling_velo_sd",
    "rolling_velo_kurt",
    "rolling_abs_rel_angle_sd",
    "rolling_abs_rel_angle_skew",
    "sinuosity_28pt",
    "est",
    "low",
    "high",
]
key_features = [f for f in key_features if f in FEATURES]

pretty_map = {
    "velo": "Velocity",
    "rolling_velo_mean": "Rolling Velocity Mean",
    "rolling_velo_sd": "Rolling Velocity SD",
    "rolling_velo_kurt": "Rolling Velocity Kurtosis",
    "rolling_abs_rel_angle_sd": "Rolling Abs Rel Angle SD",
    "rolling_abs_rel_angle_skew": "Rolling Abs Rel Angle Skew",
    "sinuosity_28pt": "Sinuosity",
    "est": "Median Space Usage",
    "low": "Low CI Space Usage",
    "high": "High CI Space Usage",
}

# =========================
# Summarize deer-level means first
# This avoids overweighting deer with more rows
# =========================
def deer_level_feature_means(df: pd.DataFrame, ids: list[int], features: list[str]) -> pd.DataFrame:
    sub = df[df["id"].isin(ids)].copy()
    return (
        sub.groupby("id")[features]
        .mean(numeric_only=True)
        .reset_index()
    )

neg_deer   = deer_level_feature_means(neg_raw, neg_raw["id"].unique().tolist(), key_features)
early_deer = deer_level_feature_means(test_raw, early_ids, key_features)
late_deer  = deer_level_feature_means(test_raw, late_ids, key_features)

group_data = {
    "Negative": neg_deer,
    "Early": early_deer,
    "Late": late_deer,
}

group_order = ["Negative", "Early", "Late"]

# =========================
# Build summary table: mean ± 95% CI
# =========================
rows = []
for feat in key_features:
    for grp in group_order:
        vals = group_data[grp][feat].dropna().to_numpy(dtype=float)
        n = len(vals)
        mean = np.mean(vals) if n > 0 else np.nan
        sd = np.std(vals, ddof=1) if n > 1 else np.nan
        se = sd / np.sqrt(n) if n > 1 else np.nan
        ci95 = 1.96 * se if n > 1 else np.nan

        rows.append({
            "feature": feat,
            "group": grp,
            "n_deer": n,
            "mean": mean,
            "sd": sd,
            "se": se,
            "ci95": ci95,
            "lower": mean - ci95 if np.isfinite(ci95) else np.nan,
            "upper": mean + ci95 if np.isfinite(ci95) else np.nan,
        })

summary_df = pd.DataFrame(rows)

print("\nDeer counts by group:")
for grp in group_order:
    print(f"{grp}: {group_data[grp]['id'].nunique()} deer")

print("\nSummary table:")
print(summary_df)

# Optional CSV output
summary_df.to_csv(f"{SAVE_BASE}_summary.csv", index=False)

# =========================
# Plot absolute raw values
# =========================
n_features = len(key_features)
ncols = 2
nrows = int(np.ceil(n_features / ncols))

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10.5,
    "ytick.labelsize": 10.5,
})

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(12.5, 3.6 * nrows),
    constrained_layout=True
)

axes = np.atleast_1d(axes).flatten()

x_positions = np.arange(len(group_order))

# Colorblind-safe palette (Okabe–Ito)
group_colors = {
    "Negative": "#0072B2",   # blue
    "Early": "#E69F00",      # orange
    "Late": "#009E73",       # green
}

line_color = "#2F2F2F"
grid_color = "#D9D9D9"

for ax, feat in zip(axes, key_features):
    sub = summary_df[summary_df["feature"] == feat].copy()
    sub["group"] = pd.Categorical(sub["group"], categories=group_order, ordered=True)
    sub = sub.sort_values("group")

    means = sub["mean"].to_numpy(dtype=float)
    errs = sub["ci95"].to_numpy(dtype=float)

    # grid
    ax.grid(True, axis="y", linestyle=":", linewidth=0.8, alpha=0.7, color=grid_color)
    ax.set_axisbelow(True)

    # plot each point separately so colors differ
    for i, grp in enumerate(group_order):
        ax.errorbar(
            x_positions[i],
            means[i],
            yerr=errs[i],
            fmt="o",
            color=line_color,
            ecolor=line_color,
            elinewidth=1.3,
            capsize=4,
            capthick=1.1,
            markersize=8,
            markerfacecolor=group_colors[grp],
            markeredgecolor="black",
            markeredgewidth=0.8,
            zorder=3
        )

    ax.set_xticks(x_positions)
    ax.set_xticklabels(group_order)
    ax.set_title(unit_map.get(feat, feat), pad=6, fontsize=11)
    unit_map = {
    "velo": "Velocity (m/s)",
    "rolling_velo_mean": "Rolling Velocity Mean (m/s)",
    "rolling_velo_sd": "Rolling Velocity SD (m/s)",
    "rolling_velo_kurt": "Rolling Velocity Kurtosis",
    "rolling_abs_rel_angle_sd": "Rolling Abs Rel Angle SD (°)",
    "rolling_abs_rel_angle_skew": "Rolling Abs Rel Angle Skew",
    "sinuosity_28pt": "Sinuosity",
    "est": "Median Space Usage (km²)",
    "low": "Low CI Space Usage (km²)",
    "high": "High CI Space Usage (km²)",
}

    # clean spines
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# remove unused axes if odd number of panels
for j in range(len(key_features), len(axes)):
    fig.delaxes(axes[j])

fig.savefig(f"{SAVE_BASE}.png", dpi=300, bbox_inches="tight")
fig.savefig(f"{SAVE_BASE}.pdf", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
<LOCAL_PATH>

# =========================
# Percent differences vs CWD-negative group
# =========================

summary_wide = summary_df.pivot(
    index="feature",
    columns="group",
    values="mean"
).reset_index()

summary_wide["early_pct_diff_vs_negative"] = (
    (summary_wide["Early"] - summary_wide["Negative"]) /
    summary_wide["Negative"]
) * 100

summary_wide["late_pct_diff_vs_negative"] = (
    (summary_wide["Late"] - summary_wide["Negative"]) /
    summary_wide["Negative"]
) * 100

summary_wide["late_pct_diff_vs_early"] = (
    (summary_wide["Late"] - summary_wide["Early"]) /
    summary_wide["Early"]
) * 100

summary_wide["feature_name"] = summary_wide["feature"].map(pretty_map).fillna(summary_wide["feature"])

pct_table = summary_wide[
    [
        "feature",
        "feature_name",
        "Negative",
        "Early",
        "Late",
        "early_pct_diff_vs_negative",
        "late_pct_diff_vs_negative",
        "late_pct_diff_vs_early",
    ]
].copy()

print("\nPercent differences based on deer-level raw means:")
print(pct_table.round(2).to_string(index=False))

pct_table.to_csv(f"{SAVE_BASE}_percent_differences.csv", index=False)
print(f"\n✅ Saved percent difference table → {SAVE_BASE}_percent_differences.csv")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

# =========================
# Paths
# =========================
BASE = Path("<LOCAL_PATH>")
UNAFFECTED_PATH = BASE / "unaffected_combined.csv"
AFFECTED_PATH   = BASE / "affected_combined.csv"

OUT_DIR = BASE / "figures_maps"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SAVE_BASE = OUT_DIR / "cwd_space_use_point_map_by_id"

# =========================
# Load data
# =========================
unaffected = pd.read_csv(UNAFFECTED_PATH)
affected   = pd.read_csv(AFFECTED_PATH)

# Keep only rows with usable coordinates
coord_x = "utm_easting"
coord_y = "utm_northing"

unaffected = unaffected.dropna(subset=[coord_x, coord_y, "id"]).copy()
affected   = affected.dropna(subset=[coord_x, coord_y, "id"]).copy()

# Make sure IDs are treated consistently
unaffected["id"] = unaffected["id"].astype(str)
affected["id"]   = affected["id"].astype(str)

# Add group labels
unaffected["group"] = "CWD−"
affected["group"]   = "CWD+"

df = pd.concat([unaffected, affected], ignore_index=True)

# =========================
# Build ID-specific color map
# =========================
def make_shades(cmap_name, n, start=0.35, end=0.95):
    cmap = plt.get_cmap(cmap_name)
    if n <= 1:
        vals = [0.7]
    else:
        vals = np.linspace(start, end, n)
    return [mcolors.to_hex(cmap(v)) for v in vals]

pos_ids = sorted(df.loc[df["group"] == "CWD+", "id"].unique())
neg_ids = sorted(df.loc[df["group"] == "CWD−", "id"].unique())

pos_colors = make_shades("Reds", len(pos_ids))
neg_colors = make_shades("Blues", len(neg_ids))

id_color_map = {}
for i, animal_id in enumerate(pos_ids):
    id_color_map[animal_id] = pos_colors[i]
for i, animal_id in enumerate(neg_ids):
    id_color_map[animal_id] = neg_colors[i]

df["plot_color"] = df["id"].map(id_color_map)

# =========================
# Plot
# =========================
fig, ax = plt.subplots(figsize=(11, 10))

for animal_id, sub in df.groupby("id", sort=False):
    ax.scatter(
        sub[coord_x],
        sub[coord_y],
        s=4,
        alpha=0.20,
        c=sub["plot_color"].iloc[0],
        edgecolors="none"
    )

# Legend proxies
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', label='CWD+ individuals',
           markerfacecolor=plt.get_cmap("Reds")(0.75), markersize=8),
    Line2D([0], [0], marker='o', color='w', label='CWD− individuals',
           markerfacecolor=plt.get_cmap("Blues")(0.75), markersize=8),
]

ax.legend(handles=legend_elements, loc="upper right", frameon=True)

ax.set_title("Animal Locations by Individual and CWD Status", fontsize=16)
ax.set_xlabel("UTM Easting", fontsize=13)
ax.set_ylabel("UTM Northing", fontsize=13)
ax.tick_params(axis="both", labelsize=11)

plt.tight_layout()
plt.savefig(f"{SAVE_BASE}.png", dpi=300, bbox_inches="tight")
plt.savefig(f"{SAVE_BASE}.pdf", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl
from pathlib import Path
import numpy as np

# === File paths ===
csv_paths = {
    "60/40": "<LOCAL_PATH>",
    "Full Holdout": "<LOCAL_PATH>",
    "10%": "<LOCAL_PATH>",
    "20%": "<LOCAL_PATH>",
    "30%": "<LOCAL_PATH>",
    "40%": "<LOCAL_PATH>",
}

model_order = ["60/40", "Full Holdout", "10%", "20%", "30%", "40%"]

# === Load & combine ===
dfs = []
for label, path in csv_paths.items():
    df = pd.read_csv(path)
    df = df[["feature", "All_%"]].copy()
    df["model"] = label
    dfs.append(df)

all_df = pd.concat(dfs, ignore_index=True)

heatmap_df = all_df.pivot(index="feature", columns="model", values="All_%")
present_cols = [m for m in model_order if m in heatmap_df.columns]
heatmap_df = heatmap_df.reindex(columns=present_cols)

pretty_order = [
    "velo", "rolling_velo_mean", "rolling_velo_sd",
    "rolling_velo_skew", "rolling_velo_kurt",
    "abs_rel_angle", "rolling_abs_rel_angle_mean",
    "rolling_abs_rel_angle_sd", "rolling_abs_rel_angle_skew",
    "rolling_abs_rel_angle_kurt",
    "sinuosity_28pt",
    "est", "low", "high"
]
present_rows = [r for r in pretty_order if r in heatmap_df.index]
heatmap_df = heatmap_df.reindex(present_rows)

pretty_map = {
    "velo": "Velocity",
    "rolling_velo_mean": "Rolling Velocity Mean",
    "rolling_velo_sd": "Rolling Velocity SD",
    "rolling_velo_skew": "Rolling Velocity Skew",
    "rolling_velo_kurt": "Rolling Velocity Kurtosis",
    "abs_rel_angle": "Absolute Relative Angle",
    "rolling_abs_rel_angle_mean": "Rolling Abs Rel Angle Mean",
    "rolling_abs_rel_angle_sd": "Rolling Abs Rel Angle SD",
    "rolling_abs_rel_angle_skew": "Rolling Abs Rel Angle Skew",
    "rolling_abs_rel_angle_kurt": "Rolling Abs Rel Angle Kurtosis",
    "sinuosity_28pt": "Sinuosity",
    "est": "Median Space Usage",
    "low": "Low CI Space Usage",
    "high": "High CI Space Usage",
}
heatmap_df.index = [pretty_map.get(r, r) for r in heatmap_df.index]

# ✅ Force numeric
heatmap_df = heatmap_df.apply(pd.to_numeric, errors="coerce")

# === Plot ===
save_base = "<LOCAL_PATH>"
Path(save_base).parent.mkdir(parents=True, exist_ok=True)

cmap = plt.cm.Blues.copy()
cmap.set_under("white")

vmin = 5.0
vmax = float(np.nanmax(heatmap_df.to_numpy(dtype=float)))
print("DEBUG vmin/vmax:", vmin, vmax)

# --- Font sizes ---
AX_LABEL_FS = 18
TICK_FS = 14
ANNOT_FS = 13
CBAR_LABEL_FS = 16
CBAR_TICK_FS = 13

# Slightly wider colorbar helps perception
fig = plt.figure(figsize=(10.8, 8.2), constrained_layout=True)
gs = fig.add_gridspec(nrows=1, ncols=2, width_ratios=[18, 1.8])
ax  = fig.add_subplot(gs[0, 0])
cax = fig.add_subplot(gs[0, 1])

norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)

hm = sns.heatmap(
    heatmap_df,
    annot=True, fmt=".1f",
    cmap=cmap,
    norm=norm,
    ax=ax,
    cbar=False,
    annot_kws={"fontsize": ANNOT_FS}
)

ax.set_xlabel("Model", fontsize=AX_LABEL_FS)
ax.set_ylabel("Feature", fontsize=AX_LABEL_FS)
ax.tick_params(axis="x", labelsize=TICK_FS)
ax.tick_params(axis="y", labelsize=TICK_FS)

# ✅ Force ticks to include the actual max (top tick = vmax)
# Use round “nice” ticks plus the exact vmax at top:
nice_ticks = [5, 15, 25, 35, 45, 55, vmax]
tick_labels = [f"{t:.0f}" for t in nice_ticks[:-1]] + [f"{vmax:.1f}"]

sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

cbar = fig.colorbar(sm, cax=cax, ticks=nice_ticks)
cbar.set_label("% of Reconstruction Error", rotation=90, labelpad=18, fontsize=CBAR_LABEL_FS)
cbar.ax.tick_params(labelsize=CBAR_TICK_FS)
cbar.set_ticklabels(tick_labels)

# Lock the color limits (extra safety)
cbar.mappable.set_clim(vmin, vmax)

# Debug what ticks actually ended up on the bar
print("DEBUG colorbar ticks:", cbar.get_ticks())

cax.yaxis.set_label_position("right")
cax.yaxis.set_ticks_position("right")

fig.savefig(f"{save_base}.png", dpi=300, bbox_inches="tight")
fig.savefig(f"{save_base}.pdf", bbox_inches="tight")
print(f"✅ Saved: {save_base}.png / {save_base}.pdf")

plt.show()
plt.close(fig)